In [0]:
# Gold Layer 생성을 위해 품질 검증이 완료된 Silver 데이터를 로드한다.
# Gold는 특정 분석 및 리포팅 목적에 맞게 데이터를 집계한 serving layer이다.

from pyspark.sql import functions as F

studies_df = spark.read.table(
    "clinical_trials.silver_studies"
)

conditions_df = spark.read.table(
    "clinical_trials.silver_conditions"
)

locations_df = spark.read.table(
    "clinical_trials.silver_locations"
)

In [0]:
# 임상시험 시작일에서 연도를 추출한다.
# Gold 집계에서는 원본 날짜보다 연도/월 등 분석 단위를 별도 컬럼으로 만드는 경우가 많다.

studies_analytics_df = (
    studies_df
    .withColumn(
        "study_year",
        F.year("start_date")
    )
)

display(
    studies_analytics_df.select(
        "nct_id",
        "start_date",
        "study_year"
    ).limit(20)
)

nct_id,start_date,study_year
NCT00634335,2008-03-01,2008
NCT05604235,2022-11-15,2022
NCT06927635,2025-07-08,2025
NCT06362135,2023-08-01,2023
NCT01805635,2013-02-01,2013
NCT01228435,2010-10-01,2010
NCT01748435,2013-02-01,2013
NCT04811235,2022-09-24,2022
NCT04208035,2018-09-17,2018
NCT04382235,2020-03-31,2020


In [0]:
# Phase는 array이지만 연도별 전체 집계에서 study 중복을 방지하기 위해
# explode하지 않고 복수 Phase를 하나의 phase_label로 표현한다. -> 1 study = 1 row 관계를 유지

studies_analytics_df = (
    studies_analytics_df
    .withColumn(
        "phase_label",
        F.when(
            F.col("phases").isNull() |
            (F.size("phases") == 0),
            F.lit("NOT_SPECIFIED")
        ).otherwise(
            F.concat_ws("/", "phases")
        )
    )
)

In [0]:
# Gold 1 : 연도별 임상시험 Summary

# 임상시험 시작연도, study type, phase, status를 기준으로 trial 수와 enrollment를 집계한다.
# 반복적인 groupBy 연산을 매번 수행하지 않고 분석/리포팅에서 바로 사용할 수 있는 Gold Table을 생성한다.

gold_yearly_summary_df = (
    studies_analytics_df
    .filter(F.col("study_year").isNotNull())
    .groupBy(
        "study_year",
        "study_type",
        "phase_label",
        "overall_status"
    )
    .agg(
        # 입력이 silver_studies이고, 이미:1 row = 1 study 를 보장하고 있기 때문에 countDistinct대신 count 사용
        F.count("nct_id").alias("study_count"),

        F.sum("enrollment_count")
            .alias("total_enrollment"),

        F.avg("enrollment_count")
            .alias("avg_enrollment")
    )
    .orderBy("study_year")
)

display(gold_yearly_summary_df)

study_year,study_type,phase_label,overall_status,study_count,total_enrollment,avg_enrollment
1900,INTERVENTIONAL,PHASE2,WITHDRAWN,1,0,0.0
1916,INTERVENTIONAL,NA,COMPLETED,1,50,50.0
1917,INTERVENTIONAL,NA,COMPLETED,1,63,63.0
1931,INTERVENTIONAL,PHASE2/PHASE3,COMPLETED,1,367,367.0
1948,OBSERVATIONAL,NOT_SPECIFIED,COMPLETED,1,null,null
1957,OBSERVATIONAL,NOT_SPECIFIED,COMPLETED,2,7259,3629.5
1958,OBSERVATIONAL,NOT_SPECIFIED,COMPLETED,1,null,null
1963,OBSERVATIONAL,NOT_SPECIFIED,COMPLETED,1,32000000,3.2E7
1964,OBSERVATIONAL,NOT_SPECIFIED,ACTIVE_NOT_RECRUITING,1,3490,3490.0
1965,OBSERVATIONAL,NOT_SPECIFIED,ACTIVE_NOT_RECRUITING,1,13000000,1.3E7


In [0]:
(
    gold_yearly_summary_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "clinical_trials.gold_yearly_summary"
    )
)

In [0]:
# Gold 2 : 국가별 임상시험 Summary
# 하나의 study가 동일 국가에 여러 location을 가질 수 있으므로
# nct_id-country 조합을 먼저 deduplicate한 후 임상시험 수를 집계한다.

# Study-level enrollment는 국가별 모집 인원을 의미하지 않으므로
# 국가별 summary에는 enrollment를 합산하지 않는다.

study_country_df = (
    locations_df
    .filter(F.col("country").isNotNull())
    .select(
        "nct_id",
        "country"
    )
    .dropDuplicates([
        "nct_id",
        "country"
    ])
)

# Study 정보와 JOIN
study_country_enriched_df = (
    study_country_df
    .join(
        studies_analytics_df.select(
            "nct_id",
            "study_year",
            "study_type",
            "overall_status"
        ),
        on="nct_id",
        how="inner"
    )
)

gold_country_summary_df = (
    study_country_enriched_df
    .filter(
        F.col("study_year").isNotNull()
    )
    .groupBy(
        "country",
        "study_year"
    )
    .agg(
        F.countDistinct("nct_id")
            .alias("trial_count"),

        F.countDistinct(
            F.when(
                F.col("study_type") == "INTERVENTIONAL",
                F.col("nct_id")
            )
        ).alias("interventional_count"),

        F.countDistinct(
            F.when(
                F.col("study_type") == "OBSERVATIONAL",
                F.col("nct_id")
            )
        ).alias("observational_count")
    )
)

display(
    gold_country_summary_df
    .orderBy(
        F.desc("trial_count")
    )
)

country,study_year,trial_count,interventional_count,observational_count
United States,2021,10771,8977,1794
United States,2019,9828,8300,1528
United States,2022,9796,8315,1481
United States,2018,9718,8310,1408
United States,2023,9714,8240,1474
United States,2020,9457,7715,1742
United States,2017,9416,7914,1502
United States,2024,9256,7935,1321
United States,2025,9142,7924,1218
United States,2016,8203,6881,1322


In [0]:
(
    gold_country_summary_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "clinical_trials.gold_country_summary"
    )
)

In [0]:
# Gold 3 : Condition Trend

condition_study_df = (
    conditions_df
    .join(
        studies_analytics_df.select(
            "nct_id",
            "study_year",
            "study_type"
        ),
        on="nct_id",
        how="inner"
    )
)

# Condition-Year 단위로 임상시험 추이를 집계한다.
#
# 1. 시작연도가 없는 study는 연도별 분석에서 제외한다.
# 2. condition과 study_year 기준으로 row를 그룹화한다.
# 3. 고유 nct_id 개수를 전체 임상시험 수로 집계한다.
# 4. 조건부 countDistinct를 이용해
#    INTERVENTIONAL / OBSERVATIONAL study 수를 각각 계산한다.
#
# 결과 grain:
# 1 row = 1 condition × 1 study_year

gold_condition_trends_df = (
    condition_study_df
    # 연도별 추이 분석이 목적이므로 시작연도를 알 수 없는 study는 집계에서 제외한다.
    .filter(
        F.col("study_year").isNotNull()
    )
    # "condition"과 "study_year"가 같은 row끼리 한 그룹으로 묶어라.
    .groupBy(
        "condition",
        "study_year"
    )
    # 앞에서 만든 각 그룹을 하나의 row로 줄이면서 숫자를 계산
    .agg(
        # countDistinct : null을 제외하고 고유한 값만 세는 것
        F.countDistinct("nct_id")
            .alias("study_count"),

        # 현재 condition-year 그룹 안에 study_type == INTERVENTIONAL인 study의 고유 NCT ID 개수를 세고 그 컬럼 이름을 interventional_count라고 한다.
        F.countDistinct(
            # when : study_type이 INTERVENTIONAL이면 nct_id를 반환해라.
            F.when(
                F.col("study_type") == "INTERVENTIONAL",
                F.col("nct_id")
            )
        ).alias("interventional_count"),

        F.countDistinct(
            F.when(
                F.col("study_type") == "OBSERVATIONAL",
                F.col("nct_id")
            )
        ).alias("observational_count")
    )
)

display(
    gold_condition_trends_df
    .orderBy(
        F.desc("study_count")
    )
    .limit(30)
)

condition,study_year,study_count,interventional_count,observational_count
COVID-19,2020,1262,748,514
Covid19,2020,1019,482,537
Covid19,2021,879,535,344
COVID-19,2022,624,406,218
COVID-19,2021,599,407,192
Healthy,2023,597,511,86
Healthy,2022,564,506,58
Healthy,2015,562,512,50
Healthy,2021,561,495,66
Healthy,2014,529,488,41


In [0]:
(
    gold_condition_trends_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "clinical_trials.gold_condition_trends"
    )
)

In [0]:
%sql

-- 연도별 trial 수
SELECT
    study_year,
    SUM(study_count) AS trials
FROM clinical_trials.gold_yearly_summary
GROUP BY study_year
ORDER BY study_year;

study_year,trials
1900,1
1916,1
1917,1
1931,1
1948,1
1957,2
1958,1
1963,1
1964,1
1965,3


In [0]:
%sql
-- 국가별 상위 trial 수
SELECT
    country,
    SUM(trial_count) AS trials
FROM clinical_trials.gold_country_summary
GROUP BY country
ORDER BY trials DESC
LIMIT 20;

country,trials
United States,193439
China,54020
France,42861
Canada,32365
United Kingdom,28830
Germany,28437
Turkey (Türkiye),26334
Italy,26094
Spain,25500
South Korea,17692


In [0]:
%sql
-- 질환별 상위 trial 수
SELECT
    condition,
    SUM(study_count) AS trials
FROM clinical_trials.gold_condition_trends
GROUP BY condition
ORDER BY trials DESC
LIMIT 20;

condition,trials
Healthy,11009
Breast Cancer,8869
Obesity,7554
Stroke,5387
Hypertension,4649
Depression,4532
Pain,4523
Prostate Cancer,4444
Cancer,3985
Heart Failure,3777
